## **Context**
As an analyst at ABC Estate Wines, we are presented with historical data encompassing the sales of different types of wines throughout the 20th century. These datasets originate from the same company but represent sales figures for distinct wine varieties. Our objective is to delve into the data, analyze trends, patterns, and factors influencing wine sales over the course of the century. By leveraging data analytics and forecasting techniques, we aim to gain actionable insights that can inform strategic decision-making and optimize sales strategies for the future

## **Objective**
The primary objective of this project is to analyze and forecast wine sales trends for the 20th century based on historical data provided by ABC Estate Wines. We aim to equip ABC Estate Wines with the necessary insights and foresight to enhance sales performance, capitalize on emerging market opportunities, and maintain a competitive edge in the wine industry..

### Import the necessary libraries.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import seaborn as sns
import statsmodels
import sklearn

### Reading the dataset in a Time Series with proper Time frequency.

In [ ]:
df = pd.read_csv('Rose.csv',parse_dates=['YearMonth'],index_col='YearMonth')
df.head()

In [ ]:
df.shape

In [ ]:
# copying data to another variable to avoid any changes to original data
data = df.copy()

### Checking the data types of the columns for the dataset.

In [ ]:
data.info()

### Checking for missing values

In [ ]:
# checking for null values
data.isnull().sum()

### Checking for Duplicates

In [ ]:
data.duplicated().sum()

In [ ]:
data.describe(include="all").T

### Imputing Missing values using Interpolation

In [ ]:
# Filter only 1994 data
data_1994 = data[data.index.year == 1994]
data_1994

In [ ]:
# Apply spline interpolation to the Rose column
data_1994["Rose"] = data_1994["Rose"].interpolate(method="spline", order=1)

In [ ]:
# Merge back with original dataset
data.update(data_1994)

In [ ]:
data.isnull().sum()

In [ ]:
data.describe(include="all").T

### Ploting the Time Series Data.

In [ ]:
from pylab import rcParams

In [ ]:
rcParams['figure.figsize'] = 15,8

In [ ]:
data.plot();
plt.grid()

### Plot a boxplot to understand the variation of Rose wine Sales to months across years.

In [ ]:
sns.boxplot(x=data.index.month,y=data['Rose'])
plt.grid();

### Plot a boxplot to understand the variation of Rose wine Sales to Years across years

In [ ]:
sns.boxplot(x=data.index.year,y=data['Rose'])
plt.grid();

### Plot a graph of monthly Rose wine Sales across years.

In [ ]:
monthly_Rose_across_years = pd.pivot_table(data, values = 'Rose', columns = data.index.month_name(), index = data.index.year)
monthly_Rose_across_years

In [ ]:
monthly_Rose_across_years.plot()
plt.grid()
plt.legend(loc='best');

### Plot a time series monthplot to understand the spread of Rose Wine Sales across different years and within different months across years.

In [ ]:
from statsmodels.graphics.tsaplots import month_plot

month_plot(data['Rose'],ylabel='Rose Wine Sales')
plt.grid();

### Plot the Empirical Cumulative Distribution of Sprakling Wine Sales.

In [ ]:
# statistics
from statsmodels.distributions.empirical_distribution import ECDF

plt.figure(figsize = (18, 8))
cdf = ECDF(data['Rose'])
plt.plot(cdf.x, cdf.y, label = "statmodels");
plt.grid()
plt.xlabel('Rose Wine Sales');
plt.ylabel('pct_of_sales');

### Plot a graph of the average and percentage change of Sales of Rose wine across years

In [ ]:
# group by date and get average Customers, and precent change
average_Rose_sales    = data.groupby(data.index)["Rose"].mean()
pct_change_in_sales = data.groupby(data.index)["Rose"].sum().pct_change()

fig, (axis1,axis2) = plt.subplots(2,1,sharex=True,figsize=(15,8))

# plot average Sales over time(year-month)
ax1 = average_Rose_sales.plot(legend=True,ax=axis1,marker='o',title="average_Rose_sales")

ax1.set_xticks(range(len(average_Rose_sales)))
ax1.set_xticklabels(average_Rose_sales.index.tolist(), rotation=90)

# plot precent change for Sales over time(year-month)
ax2 = pct_change_in_sales.plot(legend=True,ax=axis2,marker='o',rot=90,colormap="summer",title="pct_change_in_sales")

### Quarterly Plot

In [ ]:
data_quarterly_sum = data.resample('Q').sum()
data_quarterly_sum.head()

In [ ]:
data_quarterly_sum.plot();
plt.grid()

In [ ]:
data_quarterly_mean= data.resample('Q').mean()
data_quarterly_mean.head()

In [ ]:
data_quarterly_mean.plot();
plt.grid()

In [ ]:
data.plot()
plt.grid();

## Decomposing the Time Series

### Additive Model

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
decomposition = seasonal_decompose(data,model='additive')
decomposition.plot();

### From the plot We can see that alot of residuals are situated around 0 

### Let's check for Multiplicative Decompostion

In [ ]:
decomposition = seasonal_decompose(data['Rose'],model='multiplicative')
decomposition.plot();

### For the multiplicative series, we see that a lot of residuals are located around 1. Thus, Multiplicative Decomposition is the right way to decompose the time series

In [ ]:
trend = decomposition.trend
seasonality = decomposition.seasonal
residual = decomposition.resid

In [ ]:
print('Trend','\n',trend.round(2).head(12),'\n')
print('Seasonality','\n',seasonality.round(2).head(12),'\n')
print('Residual','\n',residual.round(2).head(12),'\n')

In [ ]:
deaseasonalized_ts = trend + residual
deaseasonalized_ts.round(2).head(12)

In [ ]:
data.plot()
deaseasonalized_ts.plot()
plt.legend(["Original Time Series", "Time Series without Seasonality Component"]);

### Checking for stationarity of the whole Time Series data.

In [ ]:
## Test for stationarity of the series - Dicky Fuller test

from statsmodels.tsa.stattools import adfuller
def test_stationarity(timeseries):
    
    #Determing rolling statistics
    rolmean = timeseries.rolling(window=7).mean() #determining the rolling mean
    rolstd = timeseries.rolling(window=7).std()   #determining the rolling standard deviation

    #Plot rolling statistics:
    orig = plt.plot(timeseries, color='blue',label='Original')
    mean = plt.plot(rolmean, color='red', label='Rolling Mean')
    std = plt.plot(rolstd, color='black', label = 'Rolling Std')
    plt.legend(loc='best')
    plt.title('Rolling Mean & Standard Deviation')
    plt.show(block=False)
    
    #Perform Dickey-Fuller test:
    print ('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print (dfoutput,'\n')

In [ ]:
test_stationarity(data['Rose'])

### We see that at p-value is greater than alpha value which is 0.05 or 5% significant level, This tells this Time Series is non-stationary.

### Let us take a difference of order 1 and check whether the Time Series is stationary or not.

In [ ]:
test_stationarity(data['Rose'].diff().dropna())

### We see that at $\alpha$ = 0.05 the Time Series is indeed stationary.

### Ploting the Autocorrelation function plots

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
plot_acf(data['Rose'],lags=50)
plot_acf(data['Rose'].diff().dropna(),lags=50,title='Differenced Data Autocorrelation')
plt.show()

### From the above plots, we can say that there seems to be a seasonality in the data.

## Split the data into train and test and plot the training and test data.

### Training Data is till the end of 1990. Test Data is from the beginning of 1991 to the last time stamp provided.

In [ ]:
train=data[data.index.year <= 1990]
test=data[data.index.year > 1990]

In [ ]:
print(train.shape)
print(test.shape)

In [ ]:
print(train.head(),'\n')
print(train.tail(),'\n\n')
print(test.head(),'\n')
print(test.tail(),'\n')

### Testing the training data for stationarity using the Augmented Dickey-Fuller (ADF) test at $\alpha$ = 0.05.
### Training Data stationarity for Rose Wine Sales

In [ ]:
## Test for stationarity of the series - Dicky Fuller test

from statsmodels.tsa.stattools import adfuller
def test_stationarity(timeseries):
    
    #Determing rolling statistics
    rolmean = timeseries.rolling(window=7).mean()
    rolstd = timeseries.rolling(window=7).std()

    #Plot rolling statistics:
    orig = plt.plot(timeseries, color='blue',label='Original')
    mean = plt.plot(rolmean, color='red', label='Rolling Mean')
    std = plt.plot(rolstd, color='black', label = 'Rolling Std')
    plt.legend(loc='best')
    plt.title('Rolling Mean & Standard Deviation')
    plt.show(block=False)
    
    #Perform Dickey-Fuller test:
    print ('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print (dfoutput,'\n')

In [ ]:
test_stationarity(train['Rose'])

### We see that the series is not stationary at $\alpha$ = 0.05.

In [ ]:
test_stationarity(train['Rose'].diff().dropna())

### We see that after taking a difference of order 1 the series have become stationary at $\alpha$ = 0.05.

## Building an Automated version of an <font color='blue'>ARMA model</font> for which the best parameters are selected in accordance with the lowest Akaike Information Criteria (AIC).

In [ ]:
import itertools
p = q = range(0, 5)
d= range(1)
pdq = list(itertools.product(p, d, q))
print('Some parameter combinations for the Model...')
for i in range(1,len(pdq)):
    print('Model: {}'.format(pdq[i]))

In [ ]:
# Creating an empty Dataframe with column names only
ARMA_AIC= pd.DataFrame(columns=['param', 'AIC'])
ARMA_AIC

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

for param in pdq:
    ARMA_model = ARIMA(train['Rose'].values,order=param).fit()
    print('ARIMA{} - AIC:{}'.format(param,ARMA_model.aic))
    
    new_row = pd.DataFrame({'param': [param], 'AIC': [ARMA_model.aic]})
    ARMA_AIC = pd.concat([ARMA_AIC, new_row], ignore_index=True)

In [ ]:
## Sort the above AIC values in the ascending order to get the parameters for the minimum AIC value

ARMA_AIC.sort_values(by='AIC',ascending=True)

In [ ]:
auto_ARIMA = ARIMA(train['Rose'], order=(3,0,3))

results_auto_ARIMA = auto_ARIMA.fit()

print(results_auto_ARIMA.summary())

### Predicting on the Test of Rose wine Set using this model and evaluate the model.

In [ ]:
predicted_auto_ARIMA = results_auto_ARIMA.forecast(steps=len(test))

In [ ]:
from sklearn.metrics import  mean_squared_error
rmse = mean_squared_error(test['Rose'],predicted_auto_ARIMA ,squared=False)
print(rmse)

In [ ]:
resultsDf = pd.DataFrame({'RMSE': [rmse]}
                           ,index=['ARMA(4,0,4)'])

resultsDf

## Building an Automated version of an <font color='blue'>ARIMA model</font> for which the best parameters are selected in accordance with the lowest Akaike Information Criteria (AIC).

In [ ]:
import itertools
p = q = range(0, 5)
d= range(1,3)
pdq = list(itertools.product(p, d, q))
print('Some parameter combinations for the Model...')
for i in range(1,len(pdq)):
    print('Model: {}'.format(pdq[i]))

In [ ]:
# Creating an empty Dataframe with column names only
ARIMA_AIC = pd.DataFrame(columns=['param', 'AIC'])
ARIMA_AIC

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

for param in pdq:
    ARIMA_model = ARIMA(train['Rose'].values,order=param).fit()
    print('ARIMA{} - AIC:{}'.format(param,ARIMA_model.aic))
    
    new_row = pd.DataFrame({'param': [param], 'AIC': [ARIMA_model.aic]})
    ARIMA_AIC = pd.concat([ARIMA_AIC, new_row], ignore_index=True)

In [ ]:
## Sort the above AIC values in the ascending order to get the parameters for the minimum AIC value

ARIMA_AIC.sort_values(by='AIC',ascending=True)

In [ ]:
auto_ARIMA = ARIMA(train['Rose'], order=(2,1,3))

results_auto_ARIMA = auto_ARIMA.fit()

print(results_auto_ARIMA.summary())

### Predicting on the Test Set using this model and evaluate the model.

In [ ]:
predicted_auto_ARIMA = results_auto_ARIMA.forecast(steps=len(test))

In [ ]:
from sklearn.metrics import  mean_squared_error
rmse = mean_squared_error(test['Rose'],predicted_auto_ARIMA,squared=False)
print(rmse)

In [ ]:
temp_resultsDf0 = pd.DataFrame({'RMSE': [rmse]}
                           ,index=['ARIMA(2,1,3)'])


resultsDf = pd.concat([resultsDf,temp_resultsDf0])
resultsDf

### Building an Automated version of a <font color='blue'>SARIMA</font> model for which the best parameters are selected in accordance with the lowest Akaike Information Criteria (AIC).

### SARIMA Model for Rose Wine Sales 

In [ ]:
plot_acf(data['Rose'].diff().dropna(),lags=50,title='Differenced Data Autocorrelation')
plt.show()

#### We see that there can be a seasonality of 6 for Rose Wine Sales. But from the decompostion at the start we ascertained that visually it looks like the seasonality =6 and thus using the same 

In [ ]:
import itertools
p = q = range(0, 3)
d= range(1,2)
D = range(0,1)
pdq = list(itertools.product(p, d, q))
model_pdq = [(x[0], x[1], x[2], 6) for x in list(itertools.product(p, D, q))]
print('Examples of some parameter combinations for Model...')
for i in range(1,len(pdq)):
    print('Model: {}{}'.format(pdq[i], model_pdq[i]))

In [ ]:
SARIMA_AIC = pd.DataFrame(columns=['param','seasonal', 'AIC'])
SARIMA_AIC

In [ ]:
import statsmodels.api as sm

for param in pdq:
    for param_seasonal in model_pdq:
        SARIMA_model = sm.tsa.statespace.SARIMAX(train['Rose'].values,
                                            order=param,
                                            seasonal_order=param_seasonal,
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)
            
        results_SARIMA = SARIMA_model.fit(maxiter=1000)
        print('SARIMA{}x{} - AIC:{}'.format(param, param_seasonal, results_SARIMA.aic))

        new_row = pd.DataFrame({'param':[param],'seasonal':[param_seasonal] ,'AIC': results_SARIMA.aic})
        SARIMA_AIC = pd.concat([SARIMA_AIC, new_row], ignore_index=True)

In [ ]:
SARIMA_AIC.sort_values(by=['AIC']).head()

In [ ]:
import statsmodels.api as sm

auto_SARIMA_6 = sm.tsa.statespace.SARIMAX(train['Rose'].values,
                                order=(1, 1, 2),
                                seasonal_order=(2, 0, 2, 6),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
results_auto_SARIMA_6 = auto_SARIMA_6.fit(maxiter=1000)
print(results_auto_SARIMA_6.summary())

In [ ]:
results_auto_SARIMA_6.plot_diagnostics()
plt.show()

### From the model diagnostics plot, we can see that all the individual diagnostics plots almost follow the theoretical numbers and thus we cannot develop any pattern from these plots. 

### Predict on the Test Set using this model and evaluate the model.

In [ ]:
predicted_auto_SARIMA_6 = results_auto_SARIMA_6.get_forecast(steps=len(test))

In [ ]:
predicted_auto_SARIMA_6.summary_frame(alpha=0.05).head()

In [ ]:
rmse = mean_squared_error(test['Rose'],predicted_auto_SARIMA_6.predicted_mean,squared=False)
print(rmse)

In [ ]:
temp_resultsDf = pd.DataFrame({'RMSE': [rmse]}
                           ,index=['SARIMA(1, 1, 2)(2, 0, 2, 6)'])


resultsDf = pd.concat([resultsDf,temp_resultsDf])

resultsDf

#### We see that there can be a seasonality of 12 for Rose Wine Sales. But from the decompostion at the start we ascertained that visually it looks like the seasonality =12 and thus using the same 

In [ ]:
import itertools
p = q = range(0, 3)
d= range(1,2)
D = range(0,1)
pdq = list(itertools.product(p, d, q))
model_pdq = [(x[0], x[1], x[2], 12) for x in list(itertools.product(p, D, q))]
print('Examples of some parameter combinations for Model...')
for i in range(1,len(pdq)):
    print('Model: {}{}'.format(pdq[i], model_pdq[i]))

In [ ]:
SARIMA_AIC = pd.DataFrame(columns=['param','seasonal', 'AIC'])
SARIMA_AIC

In [ ]:
import statsmodels.api as sm

for param in pdq:
    for param_seasonal in model_pdq:
        SARIMA_model = sm.tsa.statespace.SARIMAX(train['Rose'].values,
                                            order=param,
                                            seasonal_order=param_seasonal,
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)
            
        results_SARIMA = SARIMA_model.fit(maxiter=1000)
        print('SARIMA{}x{} - AIC:{}'.format(param, param_seasonal, results_SARIMA.aic))

        new_row = pd.DataFrame({'param':[param],'seasonal':[param_seasonal] ,'AIC': results_SARIMA.aic})
        SARIMA_AIC = pd.concat([SARIMA_AIC, new_row], ignore_index=True)

In [ ]:
SARIMA_AIC.sort_values(by=['AIC']).head()

In [ ]:
import statsmodels.api as sm

auto_SARIMA_12 = sm.tsa.statespace.SARIMAX(train['Rose'].values,
                                order=(0, 1, 2),
                                seasonal_order=(2, 0, 2, 12),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
results_auto_SARIMA_12 = auto_SARIMA_12.fit(maxiter=1000)
print(results_auto_SARIMA_12.summary())

In [ ]:
results_auto_SARIMA_12.plot_diagnostics()
plt.show()

### From the model diagnostics plot, we can see that all the individual diagnostics plots almost follow the theoretical numbers and thus we cannot develop any pattern from these plots. 

### Predict on the Test Set using this model and evaluate the model.

In [ ]:
predicted_auto_SARIMA_12 = results_auto_SARIMA_12.get_forecast(steps=len(test))

In [ ]:
predicted_auto_SARIMA_12.summary_frame(alpha=0.05).head()

In [ ]:
rmse = mean_squared_error(test['Rose'],predicted_auto_SARIMA_12.predicted_mean,squared=False)
print(rmse)

In [ ]:
temp_resultsDf = pd.DataFrame({'RMSE': [rmse]}
                           ,index=['SARIMA(0, 1, 2)(2,0,2,12)'])


resultsDf = pd.concat([resultsDf,temp_resultsDf])

resultsDf

## Building the most optimum model on the Rose Wine Full Data.

In [ ]:
full_data_model = sm.tsa.statespace.SARIMAX(data['Rose'],
                                order=(1,1,2),
                                seasonal_order=(2, 0, 2, 6),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
results_full_data_model = full_data_model.fit(maxiter=1000)
print(results_full_data_model.summary())

In [ ]:
results_full_data_model.plot_diagnostics();

## Evaluate the Rose Wine model on the whole and predict Entire 20th Century i.e., unitil 1999-12-01 into the future

In [ ]:
predicted_manual_SARIMA_12_full_data = results_full_data_model.get_forecast(steps=101)

In [ ]:
predicted_manual_SARIMA_12_full_data.summary_frame(alpha=0.05).head()

In [ ]:
rmse = mean_squared_error(data['Rose'],results_full_data_model.fittedvalues,squared=False)
print('RMSE of the Full Model',rmse)

In [ ]:
pred_full_manual_SARIMA_date = predicted_manual_SARIMA_12_full_data.summary_frame(alpha=0.05).set_index(pd.date_range(start='1991-08-01',end='2000-01-01', freq='M'))

In [ ]:
# plot the forecast along with the confidence band

axis = data['Rose'].plot(label='Observed')
pred_full_manual_SARIMA_date['mean'].plot(ax=axis, label='Forecast', alpha=0.8)
axis.fill_between(pred_full_manual_SARIMA_date.index, pred_full_manual_SARIMA_date['mean_ci_lower'], 
                  pred_full_manual_SARIMA_date['mean_ci_upper'], color='k', alpha=0.1)
axis.set_xlabel('YearMonth')
axis.set_ylabel('Rose')
plt.legend(loc='best')
plt.show()